In [1]:
import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning

from input import input
import config
from model import generics, hybrid_system_exp, grid_search_exp
from metrics import metrics
import numpy as np

from sklearn.neural_network import MLPRegressor
import pandas as pd

%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\functools.py' for module 'functools': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen impor

In [2]:
model = MLPRegressor(activation='logistic', solver='lbfgs') 

experiment_id = 'chamados'
model_name = f'amv1'
normalize = True
# Tarefa 3.9: force=True para regenerar -- o .pkl persistido foi treinado com
# activation='identity' (config antiga), mas o codigo acima usa 'logistic'
# desde o commit 43dae50 (2026-07-02, mudanca ja aprovada). Sem force=True,
# grid_seach_multiple_bases pularia as series ja existentes (comportamento
# intencional de idempotencia, CLAUDE.md Secao 3.2) e o .pkl continuaria
# desatualizado.
force = True
model_exec = 10

experiment_params = {
    'linear_model_name': '1arima',
    'diff_kpss': False,
    'horizon': 1
}

model_parameters = {
    'hidden_layer_sizes': [10, 20, 50], 
    'max_iter': [1000],
}

# Tarefa 3.9: lista explicita das 17 series da baseline classica, em vez de
# usar grid_seach_multiple_bases(...) (que depende de config.BASE_NAME_LIST --
# hoje com so 4 series ativas, as mesmas de FS_DEV_SERIES, por causa do foco
# recente em Feature Selection). Mesmo padrao ja usado pelos 4 notebooks de FS
# ("para nao depender/mutar config.BASE_NAME_LIST"), aplicado aqui para nao
# alterar um arquivo de configuracao compartilhado por outros notebooks so
# para esta regeneracao pontual do baseline.
baseline_series_list = [
    'airlines.txt', 'ausbee.txt', 'austres.txt', 'coloradoRiver.txt',
    'gasoline.txt', 'heartrate.txt', 'lakeerie.txt', 'lynx.txt', 'milk.txt',
    'ozon.txt', 'pollution.txt', 'redwine.txt', 'sunspot.txt', 'taylor.txt',
    'temperature.txt', 'Unemployment.txt', 'woolyrnq.txt',
]

for base_name in baseline_series_list:
    print(base_name)
    exec_gs = grid_search_exp.GridSearch(
        hybrid_system_exp.Additive,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
    )
    exec_gs.execution()


airlines.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
ausbee.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
austres.txt
{'hidden_layer_sizes': 10, 'max_iter': 1000}
coloradoRiver.txt
{'hidden_layer_sizes': 10, 'max_iter': 1000}
gasoline.txt
{'hidden_layer_sizes': 20, 'max_iter': 1000}
heartrate.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
lakeerie.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
lynx.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
milk.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
ozon.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
pollution.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
redwine.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
sunspot.txt
{'hidden_layer_sizes': 10, 'max_iter': 1000}
taylor.txt
{'hidden_layer_sizes': 20, 'max_iter': 1000}
temperature.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
Unemployment.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}
woolyrnq.txt
{'hidden_layer_sizes': 50, 'max_iter': 1000}


In [3]:
# === CELULA DE VERIFICACAO POS-EXECUCAO -- Tarefa 3.9 ===
# Roda automaticamente ao final deste "Run All" (ou pode ser reexecutada
# isoladamente depois). Confirma: (a) o .pkl regenerado persiste
# activation='logistic'; (b) comparado ao snapshot de hash pre-regeneracao,
# SO os 17 *_1amv1.pkl mudaram -- nenhum outro modelo (_1arima/_1mlp/_1svr/
# _1as) foi tocado.
import hashlib
import pickle
from pathlib import Path

chamados_dir = Path(config.ROOT_PATH) / 'data' / 'result' / 'chamados'

print("--- (a) activation persistido apos a regeneracao ---")
for base_name in ['airlines.txt', 'sunspot.txt']:
    series = base_name.replace('.txt', '')
    with open(chamados_dir / f'{series}_1amv1.pkl', 'rb') as f:
        saved = pickle.load(f)
    mlp = saved[0]['experiment'].model
    print(f"{series}: activation={mlp.activation!r}")
    assert mlp.activation == 'logistic', f"{series}: esperado 'logistic', achou {mlp.activation!r}"
print("OK -- activation='logistic' confirmado nos .pkl regenerados.")

print()
print("--- (b) comparando hashes com o snapshot pre-regeneracao ---")
snapshot_path = Path(config.ROOT_PATH) / 'data' / 'result' / 'chamados_1amv1_pkl_hashes_pre_tarefa_3_9_20260721.txt'
old_hashes = {}
for line in snapshot_path.read_text(encoding='utf-8').splitlines():
    if not line.strip():
        continue
    h, path = line.split('  ', 1)
    old_hashes[Path(path.strip()).name] = h.strip()

changed, unchanged = [], []
for pkl_path in sorted(chamados_dir.glob('*.pkl')):
    new_hash = hashlib.sha256(pkl_path.read_bytes()).hexdigest().upper()
    old_hash = old_hashes.get(pkl_path.name)
    if old_hash is None:
        print(f"AVISO: {pkl_path.name} nao estava no snapshot (arquivo novo?)")
        continue
    (changed if new_hash != old_hash else unchanged).append(pkl_path.name)

print(f"Arquivos que MUDARAM: {len(changed)}")
for name in sorted(changed):
    print(f"  {name}")
print(f"Arquivos que permaneceram IGUAIS: {len(unchanged)}")

only_1amv1_changed = all(name.endswith('_1amv1.pkl') for name in changed)
print()
if only_1amv1_changed and len(changed) == 17:
    print("OK -- exatamente os 17 *_1amv1.pkl mudaram, nada mais.")
else:
    print("ATENCAO -- o conjunto de arquivos alterados nao bate com o esperado "
          "(deveria ser exatamente os 17 *_1amv1.pkl). Revisar antes de prosseguir.")


--- (a) activation persistido apos a regeneracao ---
airlines: activation='logistic'
sunspot: activation='logistic'
OK -- activation='logistic' confirmado nos .pkl regenerados.

--- (b) comparando hashes com o snapshot pre-regeneracao ---
Arquivos que MUDARAM: 17
  Unemployment_1amv1.pkl
  airlines_1amv1.pkl
  ausbee_1amv1.pkl
  austres_1amv1.pkl
  coloradoRiver_1amv1.pkl
  gasoline_1amv1.pkl
  heartrate_1amv1.pkl
  lakeerie_1amv1.pkl
  lynx_1amv1.pkl
  milk_1amv1.pkl
  ozon_1amv1.pkl
  pollution_1amv1.pkl
  redwine_1amv1.pkl
  sunspot_1amv1.pkl
  taylor_1amv1.pkl
  temperature_1amv1.pkl
  woolyrnq_1amv1.pkl
Arquivos que permaneceram IGUAIS: 68

OK -- exatamente os 17 *_1amv1.pkl mudaram, nada mais.
